In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.DataFrame({
    "user_id": [1,2,3,4,5,6,7,8,9,10],
    "amount": [120.5, 80.0, np.nan, 300.0, 50.5, 120.5, 400.0, 90.0, 75.0, 300.0],
    "country": ["PL", "DE", "PL", "FR", None, "PL", "DE", "FR", "PL", "DE"],
    "signup_date": [
        "2024-01-10","2024-01-15","2024-02-01","2024-02-10","2024-03-05",
        "2024-03-08","2024-03-12","2024-04-01","2024-04-10","2024-04-20"
    ]
})

df["signup_date"] = pd.to_datetime(df["signup_date"])

In [3]:
### 1. Inspect data quality
df.info()
# df.shape
# df.describe()
# df.isna().sum()
# df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      10 non-null     int64         
 1   amount       9 non-null      float64       
 2   country      9 non-null      object        
 3   signup_date  10 non-null     datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 452.0+ bytes


In [4]:
# detect outliers
threshold = df["amount"].quantile(0.95)
df["outlier"] = df["amount"] > threshold
df[df["outlier"] == True].head()

,user_id,amount,country,signup_date,outlier
6,7,400.0,DE,2024-03-12,True


In [5]:
### 2. Fill missing values + normalization
# Replace missing amount with median and missing country with "Unknown"
df["amount"] = df["amount"].fillna(df["amount"].median())
df["country"] = df["country"].fillna("Unknown")
# Normalize text values
df["country"] = df["country"].str.strip().str.lower()
chc = df["country"].unique().tolist()
print(sorted(chc))

['de', 'fr', 'pl', 'unknown']


In [6]:
df.groupby("country")["amount"].sum().reset_index()

,country,amount
0,de,780.0
1,fr,390.0
2,pl,436.5
3,unknown,50.5


In [7]:
df.groupby("country").agg(
    avg_amount=("amount", "mean"),
    count_amount=("amount", "count")
).reset_index()

,country,avg_amount,count_amount
0,de,260.000,3
1,fr,195.000,2
2,pl,109.125,4
3,unknown,50.500,1


In [8]:
# How many users signed up each month?
df["month"] = df["signup_date"].dt.to_period("M")
df.groupby("month")["user_id"].count().reset_index()

,month,user_id
0,2024-01,2
1,2024-02,2
2,2024-03,3
3,2024-04,3


In [9]:
### 4. Creating columns
# segment column on condition
df["segment"] = "Low"
df.loc[df["amount"] >= 100, "segment"] = "Medium"
df.loc[df["amount"] >= 200, "segment"] = "High"
df.groupby("segment")["user_id"].count().reset_index()

,segment,user_id
0,High,3
1,Low,4
2,Medium,3


In [10]:
# running total amount
df["running_total"] = df["amount"].fillna(0).cumsum()
print(df[["user_id", "country", "amount", "running_total"]])

   user_id  country  amount  running_total
0        1       pl   120.5          120.5
1        2       de    80.0          200.5
2        3       pl   120.5          321.0
3        4       fr   300.0          621.0
4        5  unknown    50.5          671.5
5        6       pl   120.5          792.0
6        7       de   400.0         1192.0
7        8       fr    90.0         1282.0
8        9       pl    75.0         1357.0
9       10       de   300.0         1657.0


In [11]:
df["roll_2user_country"] = (
    df.groupby("country")["amount"]
      .rolling(2)
      .sum()
      .reset_index(level=0, drop=True)
)
print(df[["user_id", "country", "amount", "roll_2user_country"]])

   user_id  country  amount  roll_2user_country
0        1       pl   120.5                 NaN
1        2       de    80.0                 NaN
2        3       pl   120.5               241.0
3        4       fr   300.0                 NaN
4        5  unknown    50.5                 NaN
5        6       pl   120.5               241.0
6        7       de   400.0               480.0
7        8       fr    90.0               390.0
8        9       pl    75.0               195.5
9       10       de   300.0               700.0


In [12]:
# days since signup
df["days_since_signup"] = (
    pd.Timestamp.today() - df["signup_date"]
).dt.days
print(df[["user_id", "country", "signup_date", "days_since_signup"]])

   user_id  country signup_date  days_since_signup
0        1       pl  2024-01-10                838
1        2       de  2024-01-15                833
2        3       pl  2024-02-01                816
3        4       fr  2024-02-10                807
4        5  unknown  2024-03-05                783
5        6       pl  2024-03-08                780
6        7       de  2024-03-12                776
7        8       fr  2024-04-01                756
8        9       pl  2024-04-10                747
9       10       de  2024-04-20                737


In [13]:
# rank higherst amount per country
df["rank_in_country"] = (
    df.groupby("country")["amount"]
      .rank(method="dense", ascending=False)
)
print(df[["user_id", "country", "amount", "rank_in_country"]])

   user_id  country  amount  rank_in_country
0        1       pl   120.5              1.0
1        2       de    80.0              3.0
2        3       pl   120.5              1.0
3        4       fr   300.0              1.0
4        5  unknown    50.5              1.0
5        6       pl   120.5              1.0
6        7       de   400.0              1.0
7        8       fr    90.0              2.0
8        9       pl    75.0              2.0
9       10       de   300.0              2.0


In [14]:
### 5. Merging dataframes
plans = pd.DataFrame({
    "user_id": [1,2,3,4,5],
    "plan": ["Free","Premium","Free","Metal","Free"]
})
df = df.merge(plans, on="user_id", how="left")
print(df[["user_id", "country", "amount", "plan"]])

   user_id  country  amount     plan
0        1       pl   120.5     Free
1        2       de    80.0  Premium
2        3       pl   120.5     Free
3        4       fr   300.0    Metal
4        5  unknown    50.5     Free
5        6       pl   120.5      NaN
6        7       de   400.0      NaN
7        8       fr    90.0      NaN
8        9       pl    75.0      NaN
9       10       de   300.0      NaN


In [15]:
### 6. Pivot for additional dataframe
# Users per country pivot
pd.pivot_table(
    df,
    values="user_id",
    index="country",
    aggfunc="count"
)

,user_id
country,
de,3
fr,2
pl,4
unknown,1
